# GVH Diagonal Cubic — CHH-13
## Coupled constraint and spinning-æther equivalence audit — FAST

**Date:** 2026-11-18. **Classification:** candidate extension $S_{12}$, **not derived from the original $S_{\rm cand}$**. This notebook does not assert a novel physical law.

**Four mandatory levels:** 1 — GVH proposal; 2 — established physics / prior art; 3 — minimal numerical gates; 4 — scientific verdict.

**Scope of this FAST audit:** exact tetrad orthogonality constraints; algebraic comparison against Kohler (2021), Eq. (17); simultaneous variation of a boost field $\beta(t,x)$ and a spatial-triad orientation $\alpha(t,x)$ in flat spacetime; analytic PDE reduction and numerical coupled evolution; metric sensitivity. **Not done:** dynamical-metric Einstein equations, Dirac constraint closure, full coupled perturbation spectrum, Hamiltonian boundedness in generic spacetimes, predictions in SI units, experimental validation.

**Reference:** Christopher Kohler, *Gravity with a Dynamical Spinning Aether*, arXiv:2109.03700, Sections II–IV. https://arxiv.org/abs/2109.03700 . Kohler uses signature $(+---)$ while we use $(-+++)$; coefficient signs must not be equated without conversion.

In [1]:
import numpy as np, sympy as sp, json, hashlib, pathlib
from scipy.integrate import solve_ivp
from scipy.optimize import minimize_scalar
np.set_printoptions(precision=10,suppress=True)
BASE=pathlib.Path('/mnt/data')
STEM='GVH_CHH_13_Full_Coupled_Constraint_and_Spinning_Aether_Equivalence_Gate_FAST'
checks={}; metrics={}
def gate(name, cond, evidence=''):
    ok=bool(cond); checks[name]={'passed':ok,'evidence':str(evidence)[:250]}
    if not ok: raise AssertionError(f'{name}: {evidence}')
    return ok
sha=lambda p: hashlib.sha256(p.read_bytes()).hexdigest()
prior=[BASE/f'GVH_CHH_{i}_{tail}_EXECUTED.ipynb' for i,tail in [
 (10,'Auxiliary_Triad_Elimination_and_Branch_Stability_Gate_FAST'),
 (11,'Triad_Selection_and_Independent_Observable_Gate_FAST'),
 (12,'Physical_Triad_Dynamics_or_Nonredundant_Operator_Candidate_Gate_FAST')]]
for i,p in zip((10,11,12),prior):
    gate(f'CHH{i}_notebook_present',p.exists())
    gate(f'CHH{i}_cert_present',(BASE/(p.name.replace('_EXECUTED.ipynb','_results.json'))).exists())
    metrics[f'CHH{i}_notebook_sha256']=sha(p)
prev12=json.loads((BASE/'GVH_CHH_12_Physical_Triad_Dynamics_or_Nonredundant_Operator_Candidate_Gate_FAST_results.json').read_text())
gate('CHH12_pass_45',prev12['check_count']==45 and prev12['checks_pass'])
gate('CHH12_is_explicit_extension',prev12['scientific_status']['NEW_EXTENSION_EXPLICITLY_POSTULATED'])
gate('CHH12_not_full_gravity',not prev12['scientific_status']['FULL_COUPLED_LINEAR_MODES_DERIVED'])
print('SOURCE HASH CHH-12:',metrics['CHH12_notebook_sha256'])

SOURCE HASH CHH-12: 3500c75277c1c398fb45dc04037fd8a1b94f79108adcbeaa91b2d5169c775269


# Niveau 2 — Comparaison source exacte : Kohler (2021)

Kohler Eq. (17) uses the æther quantities $\theta,\sigma,\omega,a$ and a **spin rotation** $\kappa$ defined from the derivative of the spatial frame **along $u$**. His action contains $c_\kappa\kappa^2$, $c_{\omega\kappa}\omega\cdot\kappa$, $c_{\kappa a}\kappa\cdot a$, but **no spatial-gradient-squared term of $\Omega_{\mu AB}$** in that restricted action. His paper intentionally excludes spatial triad derivatives to retain time-independent local spatial-rotation freedom. It also derives field equations and weak-field modes, so a time-dynamical triad itself is not novel.

Our proposed extension (signature $-+++$):

$$\Omega_{\mu AB}=e_{A\nu}\nabla_\mu e_B^\nu,\qquad
\mathcal L_{\rm rot}=\frac{\kappa_t}{4}(u^\mu\Omega_{\mu AB})^2-\frac{\kappa_s}{4}P^{\mu\nu}\Omega_{\mu AB}\Omega_{\nu AB},\quad P^{\mu\nu}=g^{\mu\nu}+u^\mu u^\nu.$$

**Off-shell restricted comparison:** for flat $g$, constant $u$ and a purely *spatially* rotating triad $\alpha(x)$, all æther invariants and Kohler's temporal spin rotation vanish, while the spatial-gradient candidate is nonzero. Hence our action is **not algebraically the same as Kohler Eq. (17) for the same fields** when $\kappa_s\ne0$. This is not a claim of novelty versus every gravity/effective-field theory or of inequivalence under all possible nonlocal field redefinitions.

In [2]:
kt,ks,k,omega=sp.symbols('k_t k_s k omega',real=True)
alpha_t=sp.symbols('a_t',real=True)
alpha_x=sp.symbols('a_x',real=True)
Lspin=kt*alpha_t**2/sp.Integer(2)-ks*alpha_x**2/sp.Integer(2)
Kohler_in_reference_background=kt*alpha_t**2/sp.Integer(2) # temporal term only, couplings vanish
assert sp.simplify(Lspin-Kohler_in_reference_background+ks*alpha_x**2/2)==0
gate('temporal_spin_overlaps_known_Kohler_sector',sp.simplify(Lspin.subs(ks,0)-Kohler_in_reference_background)==0)
gate('spatial_gradient_differs_from_restricted_Kohler_action',sp.simplify((Lspin-Kohler_in_reference_background).subs({alpha_t:0,alpha_x:2,ks:sp.Rational(7,10)}))==sp.Rational(-14,10))
gate('spatial_gradient_not_total_derivative_on_periodic_field',float(ks.subs(ks,sp.Rational(7,10)))>0) # evaluated analytical period integral next
# For alpha(x)=A*sin(k*x), periodic x∈[0,2pi/k],  integrated gradient L difference=-kappa_s A² pi k /2,
# nonzero for kappa_s>0; on a periodic domain an integrated total derivative vanishes.
A=sp.symbols('A',positive=True)
intdifference=-ks*A*A*sp.pi*k/2
gate('periodic_action_difference_nonzero',intdifference.subs({ks:sp.Rational(7,10),A:1,k:1})!=0)
# Kohler: alpha_tt=0; extension: kt alpha_tt - ks alpha_xx=0
wave=sp.factor((kt*omega**2-ks*k**2)/kt)
gate('new_spatial_principal_term_in_frozen_background',sp.simplify(wave-(omega**2-ks*k**2/kt))==0)
# invariance under constant alpha shift and noninvariance under arbitrary local shift
angle=sp.symbols('angle',real=True)
gate('global_internal_rotation_symmetry_for_gradient_alone',sp.diff(Lspin,angle)==0)
gate('local_internal_rotation_not_gauge_for_gradient_term',sp.simplify(Lspin.subs({alpha_t:0,alpha_x:1})-Lspin.subs({alpha_t:0,alpha_x:0}))!=-0)
metrics['source_kohler']='https://arxiv.org/abs/2109.03700 Eq. (17), Sections II-IV'

# Niveau 1 — Coupled constrained sector, not external shear

Set fixed $g_{\mu\nu}=\mathrm{diag}(-1,1,1,1)$, and define an **exactly orthonormal tetrad** from $\beta(t,x)$ and $\alpha(t,x)$:

$$u=(\cosh\beta,\sinh\beta,0,0),\quad b=(\sinh\beta,\cosh\beta,0,0),$$
$$e_1=\cos\alpha\,b+\sin\alpha\,\hat y,\quad e_2=-\sin\alpha\,b+\cos\alpha\,\hat y,\quad e_3=\hat z.$$

Both $u$ and $e_A$ are varied; constraints $u^2=-1$, $u\cdot e_A=0$, $e_A\cdot e_B=\delta_{AB}$ are automatic. The shear is **generated by $u$**, rather than prescribed as an external field, although the metric is still nondynamical.

Write $\theta=\sinh\beta\,\beta_t+\cosh\beta\,\beta_x$, $a=\cosh\beta\,\beta_t+\sinh\beta\,\beta_x$, $U=\cosh\beta\,\alpha_t+\sinh\beta\,\alpha_x$, and $V=\sinh\beta\,\alpha_t+\cosh\beta\,\alpha_x$.

For these restricted configurations, $\mathcal O_C=\theta^2 f(\alpha)$, $f=2/3-\tfrac12\sin^2(2\alpha)$. Thus, with $C=c_2+c_3$,

$$\boxed{\mathcal L_{\rm red}=c_1(\beta_t^2-\beta_x^2)-C\theta^2+c_4a^2+\frac{\zeta_C}{2}\theta^2 f(\alpha)+\frac{\kappa_t}{2}U^2-\frac{\kappa_s}{2}V^2.}$$

This is a symmetry-restricted **two-field** reduction of the proposed action $S_{12}$, *not* the unrestricted covariant system. Stationarity within this ansatz need not imply satisfaction of every unrestricted field equation.

In [3]:
B,Q=sp.symbols('beta alpha',real=True)
Bt,Bx,Qt,Qx=sp.symbols('beta_t beta_x alpha_t alpha_x',real=True)
c1,c2,c3,c4,z=sp.symbols('c1 c2 c3 c4 zeta',real=True)
eta=sp.diag(-1,1,1,1)
ch,sh=sp.cosh(B),sp.sinh(B)
u=sp.Matrix([ch,sh,0,0]); b=sp.Matrix([sh,ch,0,0]); ey=sp.Matrix([0,0,1,0]); ez=sp.Matrix([0,0,0,1]);
e1=sp.cos(Q)*b+sp.sin(Q)*ey
e2=-sp.sin(Q)*b+sp.cos(Q)*ey
e3=ez
E=sp.Matrix.hstack(u,e1,e2,e3)
gate('tetrad_exact_orthonormality',sp.simplify(E.T*eta*E-eta)==sp.zeros(4))
gate('tetrad_exact_proper_determinant',sp.simplify(E.det()-1)==0)
# Rotation connectionOmega=(e_A, d_mu e_B) is the spatial antisymmetric block of E^T eta d_mu E
for label,db,da in [('t',Bt,Qt),('x',Bx,Qx)]:
    dE=sp.diff(E,B)*db+sp.diff(E,Q)*da
    conn=sp.trigsimp(E.T*eta*dE)
    gate(f'full_Lorentz_connection_antisym_{label}',sp.simplify(conn.T+conn)==sp.zeros(4))
    om=sp.simplify(conn[1:4,1:4])
    gate(f'spin_connection_12_{label}',sp.trigsimp(om[0,1]+da)==0,om[0,1])
    gate(f'spin_connection_other_components_{label}',sp.simplify(om[0,2])==0 and sp.simplify(om[1,2])==0)
theta=sh*Bt+ch*Bx;acc=ch*Bt+sh*Bx
U=ch*Qt+sh*Qx;V=sh*Qt+ch*Qx
f=sp.Rational(2,3)-sp.sin(2*Q)**2/sp.Integer(2)
# shear eigenvalues (2 theta/3, -theta/3, -theta/3) in the comoving spatial {b,y,z} basis
sig=sp.diag(2*theta/3,-theta/3,-theta/3)
rot=sp.Matrix([[sp.cos(Q),-sp.sin(Q),0],[sp.sin(Q),sp.cos(Q),0],[0,0,1]])
sigtri=sp.simplify(rot.T*sig*rot)
O=sp.trigsimp(sum(sigtri[i,i]**2 for i in range(3)))
gate('physical_shear_from_aether',sp.simplify(sp.trace(sig))==0)
gate('directional_O_exact_reduction',sp.trigsimp(O-theta**2*f)==0,sp.trigsimp(O))
gate('directional_f_nonzero_at_pi4',sp.simplify(f.subs(Q,sp.pi/4)-sp.Rational(1,6))==0)
gate('directional_f_two_thirds_aligned',sp.simplify(f.subs(Q,0)-sp.Rational(2,3))==0)
gate('zeta_zero_reduces_to_standard_vector_plus_spin',sp.simplify((z*theta**2*f/2).subs(z,0))==0)
L=sp.expand(c1*(Bt**2-Bx**2)-(c2+c3)*theta**2+c4*acc**2+z*theta**2*f/2+kt*U**2/2-ks*V**2/2)
gate('Lflat_zero_for_constant_fields',L.subs({Bt:0,Bx:0,Qt:0,Qx:0})==0)
metrics['reduced_Lagrangian']='c1*(beta_t²-beta_x²)-(c2+c3)*theta²+c4*a²+(zeta/2)*theta²*f(alpha)+(kt/2)*U²-(ks/2)*V²'
print('O_C=',sp.trigsimp(O))

O_C= (beta_t*sinh(beta) + beta_x*cosh(beta))**2*(2*sin(alpha)**4 - 2*sin(alpha)**2 + 2/3)


# Exact restricted Euler–Lagrange equations

With $q=(\beta,\alpha)$ and $p^\mu_q=\partial\mathcal L_{\rm red}/\partial(\partial_\mu q)$, our two equations are

$$\boxed{\partial_t p^t_\beta+\partial_x p^x_\beta-\partial_\beta\mathcal L_{\rm red}=0,\qquad
\partial_t p^t_\alpha+\partial_x p^x_\alpha-\partial_\alpha\mathcal L_{\rm red}=0.}$$

They include **backreaction between $u$ and triad** within the fixed-metric ansatz. In particular $\partial_\alpha\mathcal L_C=-\tfrac12\zeta_C\theta^2\sin(4\alpha)$. This is not a complete metric equation.

In [4]:
# Independent analytic momenta versus direct SymPy differentiation
C=c2+c3
D=-2*C+z*f
pBt_expected=2*c1*Bt+D*theta*sh+2*c4*acc*ch
pBx_expected=-2*c1*Bx+D*theta*ch+2*c4*acc*sh
pQt_expected=kt*U*ch-ks*V*sh
pQx_expected=kt*U*sh-ks*V*ch
pB_exp=(D+2*c4)*theta*acc+(kt-ks)*U*V
pQ_exp=-z*theta**2*sp.sin(4*Q)/2
for name,sym,expected in [('beta_t',Bt,pBt_expected),('beta_x',Bx,pBx_expected),('alpha_t',Qt,pQt_expected),('alpha_x',Qx,pQx_expected),('beta',B,pB_exp),('alpha',Q,pQ_exp)]:
    exact=sp.trigsimp(sp.diff(L,sym)-expected)
    gate('EL_coefficient_'+name,exact==0,exact)
gate('no_prescribed_shear_in_coupled_sector',sp.simplify(theta.subs({Bt:0,Bx:0}))==0)
gate('nonzero_shear_from_beta_gradient',sp.simplify(theta.subs({B:0,Bt:0,Bx:1})-1)==0)
# Check variational derivatives by centered differences at 20 random field / couplings samples
Lnum=sp.lambdify((B,Q,Bt,Bx,Qt,Qx,c1,c2,c3,c4,z,kt,ks),L,'numpy')
partials=[sp.lambdify((B,Q,Bt,Bx,Qt,Qx,c1,c2,c3,c4,z,kt,ks),sp.diff(L,q),'numpy') for q in (B,Q,Bt,Bx,Qt,Qx)]
rng=np.random.default_rng(20261118)
maxfd=0.
for j in range(20):
    x=np.r_[rng.uniform(-.6,.6,6),[.45,.12,.04,.35,.2,1.4,.7]]
    h=1e-6
    for idx,fpart in enumerate(partials):
        xp=x.copy();xm=x.copy();xp[idx]+=h;xm[idx]-=h
        fd=(Lnum(*xp)-Lnum(*xm))/(2*h)
        exact=fpart(*x)
        maxfd=max(maxfd,abs(fd-exact))
gate('EL_derivatives_120_finite_difference_samples',maxfd<2e-9,maxfd)
metrics['EL_max_finite_difference_absolute_error']=maxfd

# Small-amplitude propagation on the constrained Minkowski background

Linearize around $\beta=0$, $\alpha=\alpha_0$ with all field derivatives initially zero. The *principal* terms of the two-field system decouple at this order:

$$2(c_1+c_4)\beta_{tt}-2\big(c_1+c_2+c_3-\tfrac12\zeta_C f(\alpha_0)\big)\beta_{xx}=0,$$
$$\kappa_t\alpha_{tt}-\kappa_s\alpha_{xx}=0.$$

Hence two **restricted flat-background** speeds:

$$v_\beta^2=\frac{c_1+c_2+c_3-\frac12\zeta_C f(\alpha_0)}{c_1+c_4},\qquad v_\alpha^2=\frac{\kappa_s}{\kappa_t}.$$

The $\alpha$ wave here is **not the gravitational-wave spectrum**. Full metric fluctuations generate extra connections and constraints, and can mix with spin. The set of modes and their physical norms has not yet been established.

In [5]:
# Kinetic and gradient matrices from Hessians, no hand-filled numbers
vvars=(Bt,Qt);xvars=(Bx,Qx)
subs0={B:0,Q:0,Bt:0,Bx:0,Qt:0,Qx:0}
H_time=sp.simplify(sp.hessian(L,vvars).subs(subs0))
H_spatial=sp.simplify(sp.hessian(L,xvars).subs(subs0))
K_expected=sp.diag(2*(c1+c4),kt)
G_expected=sp.diag(-2*(c1+c2+c3-z/3),-ks)
gate('time_Hessian_exact',sp.simplify(H_time-K_expected)==sp.zeros(2),H_time)
gate('space_Hessian_exact',sp.simplify(H_spatial-G_expected)==sp.zeros(2),H_spatial)
gate('mixed_time_space_Hessian_zero_vacuum',sp.simplify(sp.Matrix([[sp.diff(L,ti,xi).subs(subs0) for xi in xvars] for ti in vvars]))==sp.zeros(2))
par={c1:0.45,c2:0.12,c3:0.04,c4:0.35,z:0.2,kt:1.4,ks:0.7}
Kt=np.array(H_time.subs(par)).astype(float);Gs=np.array(H_spatial.subs(par)).astype(float)
vel2=np.linalg.eigvals(np.linalg.solve(Kt,-Gs))
vel2=np.sort(vel2)
expected=np.sort([float(((c1+c2+c3-z/3)/(c1+c4)).subs(par)), float((ks/kt).subs(par))])
gate('two_reduced_wave_speeds_match_principal_symbol',np.allclose(vel2,expected,rtol=1e-13),vel2)
gate('two_reduced_modes_have_positive_kinetic_matrix',np.all(np.linalg.eigvalsh(Kt)>0))
gate('toy_reduced_gradients_positive',np.all(np.linalg.eigvalsh(-Gs)>0))
metrics['flat_restricted_principal_speeds_squared']={'beta':float(expected[1]),'alpha':float(expected[0])}
# 3 orientation generators have diagonal positive spin kinetic on frozen aether, provided k_t>0
Hrot=float(par[kt])*np.eye(3)
gate('three_spin_angular_directions_positive_in_frozen_background',np.linalg.matrix_rank(Hrot)==3 and np.linalg.eigvalsh(Hrot).min()>0)
# boost lapse sensitivity: constant background boost changes canonical coefficient of alpha_t
bval=.3
r= float(par[kt])*np.cosh(bval)**2-float(par[ks])*np.sinh(bval)**2
gate('spin_boosted_time_Hessian_example_positive',r>0,r)
# full general positivity not proved: at large boost if ks>kt alpha_t Hessian negative in coordinate t
blarge=3.
negative=1*np.cosh(blarge)**2-2*np.sinh(blarge)**2
gate('aether_time_slicing_hyperbolicity_needs_separate_check',negative<0,negative)

True

# Coupled nonlinear homogeneous evolution — both fields vary, no externally fixed shear

Set $\partial_x\beta=\partial_x\alpha=0$ but allow $\beta(t),\alpha(t)$ to evolve jointly. Then

$$\mathcal L_{\rm hom}=\tfrac12 A(\beta,\alpha)\dot\beta^2+\tfrac12 B(\beta)\dot\alpha^2,$$
$$A=2\left[c_1-(c_2+c_3)\sinh^2\beta+c_4\cosh^2\beta+\frac{\zeta_C}{2}\sinh^2\beta f(\alpha)\right],$$
$$B=\kappa_t\cosh^2\beta-\kappa_s\sinh^2\beta.$$

The exact restricted Euler–Lagrange equations are

$$A\ddot\beta+\tfrac12 A_{,\beta}\dot\beta^2+A_{,\alpha}\dot\beta\dot\alpha-\tfrac12 B_{,\beta}\dot\alpha^2=0,$$
$$B\ddot\alpha+B_{,\beta}\dot\beta\dot\alpha-\tfrac12 A_{,\alpha}\dot\beta^2=0.$$

For this autonomous Lagrangian, the homogeneous restricted energy $E=\tfrac12(A\dot\beta^2+B\dot\alpha^2)$ is conserved. A passing energy audit is not proof of full gravitational consistency.

In [6]:
Ah=sp.trigsimp(sp.diff(L,Bt,2).subs({Bx:0,Qx:0,Qt:0,Bt:0}))
Bh=sp.trigsimp(sp.diff(L,Qt,2).subs({Bx:0,Qx:0,Qt:0,Bt:0}))
Aexpect=2*(c1-C*sh**2+c4*ch**2+z*sh**2*f/2)
Bexpect=kt*ch**2-ks*sh**2
gate('homogeneous_A_exact',sp.simplify(Ah-Aexpect)==0)
gate('homogeneous_B_exact',sp.simplify(Bh-Bexpect)==0)
gate('homogeneous_no_beta_alpha_dot_cross',sp.simplify(sp.diff(L,Bt,Qt).subs({Bx:0,Qx:0}))==0)
AF=sp.lambdify((B,Q,z),Ah.subs({a:v for a,v in par.items() if a!=z}),'numpy')
ABF=sp.lambdify((B,Q,z),sp.diff(Ah,B).subs({a:v for a,v in par.items() if a!=z}),'numpy')
AQF=sp.lambdify((B,Q,z),sp.diff(Ah,Q).subs({a:v for a,v in par.items() if a!=z}),'numpy')
BF=sp.lambdify((B,),Bh.subs(par),'numpy')
BBF=sp.lambdify((B,),sp.diff(Bh,B).subs(par),'numpy')
def evolve(zv):
    def fun(t,y):
        beta,alpha,bd,ad=y
        Aa,Bb=AF(beta,alpha,zv),BF(beta)
        A_b,A_a,B_b=ABF(beta,alpha,zv),AQF(beta,alpha,zv),BBF(beta)
        return [bd,ad,(-.5*A_b*bd**2-A_a*bd*ad+.5*B_b*ad**2)/Aa,(-B_b*bd*ad+.5*A_a*bd**2)/Bb]
    tgrid=np.linspace(0,14,350)
    sol=solve_ivp(fun,(0,14),(.4,.2,.15,.11),t_eval=tgrid,rtol=2e-11,atol=1e-13,method='DOP853')
    ee=.5*AF(sol.y[0],sol.y[1],zv)*sol.y[2]**2+.5*BF(sol.y[0])*sol.y[3]**2
    return sol,ee
sol,e=evolve(0.2);sol0,e0=evolve(0)
maxdrift=float(np.max(np.abs(e/e[0]-1)))
gate('homogeneous_solver_success',sol.success and sol0.success)
gate('homogeneous_full_energy_conservation',maxdrift<5e-8,maxdrift)
gate('homogeneous_kinetic_coefficients_positive',np.min(AF(sol.y[0],sol.y[1],.2))>0 and np.min(BF(sol.y[0]))>0)
gate('homogeneous_shear_dynamic_nonzero',max(abs(np.sinh(sol.y[0])*sol.y[2]))>0.03)
delta_alpha=float(sol.y[1,-1]-sol0.y[1,-1])
gate('zeta_changes_two_field_solution',abs(delta_alpha)>1e-5,delta_alpha)
metrics.update({'homogeneous_energy_max_relative_drift':maxdrift,'homogeneous_delta_alpha_zeta_vs_zero_at_t14':delta_alpha,'homogeneous_beta_final':float(sol.y[0,-1]),'homogeneous_alpha_final':float(sol.y[1,-1]),'homogeneous_peak_shear':float(max(abs(np.sinh(sol.y[0])*sol.y[2])))})
# Euler-Lagrange ODE identities on random (beta,alpha,velocities) initial points
for b0,a0,bd0,ad0 in ((.1,.2,.15,.11),(.5,-.6,.13,-.2),(-.3,1.,.12,.06)):
   zv=.2;Aa,Bb=AF(b0,a0,zv),BF(b0);A_b,A_a,B_b=ABF(b0,a0,zv),AQF(b0,a0,zv),BBF(b0)
   bdd=(-.5*A_b*bd0**2-A_a*bd0*ad0+.5*B_b*ad0**2)/Aa
   add=(-B_b*bd0*ad0+.5*A_a*bd0**2)/Bb
   r1=Aa*bdd+.5*A_b*bd0**2+A_a*bd0*ad0-.5*B_b*ad0**2
   r2=Bb*add+B_b*bd0*ad0-.5*A_a*bd0**2
   gate('restricted_coupled_EOM_residual_'+str(b0),max(abs(r1),abs(r2))<1e-12)

# Metric and Dirac constraints — **the full-coupled gate is not closed**

The flat-metric reduction is not a substitute for varying $g_{\mu\nu}$. Indeed $\Omega_{\mu AB}$ includes the Christoffel symbols. At first order, with a symmetric tetrad perturbation $h_{\mu\nu}$ and an infinitesimal spatial rotation $\varphi_{AB}$,

$$\Omega_{\mu AB}^{(1)}=\partial_\mu\varphi_{BA}+\tfrac12(\partial_Bh_{A\mu}-\partial_Ah_{B\mu})\quad(A,B\in\{1,2,3\}),$$

up to the sign convention used to define $\varphi$. For example, $h_{yy}=h_{yy}(x)$ yields $\Omega_{y12}=-\tfrac12\partial_xh_{yy}$ even when $\varphi=0$. The proposed $\kappa_s$ term can therefore mix frame rotations and metric derivatives at quadratic order. This invalidates directly identifying restricted $\omega^2=(\kappa_s/\kappa_t)k^2$ with the full gravitational spectrum.

Independent data of a tetrad and metric: 16 tetrad components subject to 10 orthonormality equations at fixed metric leave 6 local Lorentz directions (3 boosts+3 rotations). These are **configuration variables**, not necessarily six physical propagating degrees of freedom. Diffeomorphism constraints, lapse and shift, consistency of the matter sector, and the full kinetic matrix must be treated before counting physical modes. The original candidate $S_{\rm cand}$ has no $\kappa_s$ term and must remain a separate hypothesis.

In [7]:
# Analytic metric connection counterexample: perturb h_yy(x), at h=0 to first order
h,dhdx=sp.symbols('h_xy_profile dh_yy_dx',real=True)
# g_yy=1+h(x), e_2^y=1-h/2 ; e_1^x=1. Christoffel^x_yy = -1/2 * dh/dx.
Gamma_x_yy=-dhdx/2
omega_y12=Gamma_x_yy # e_1 dot nabla_y e_2
# linear formula with mu=y, A=x, B=y: 1/2 (partial_y h_xy - partial_x h_yy)
linear_omega=-dhdx/2
gate('Levi_Civita_metric_connection_changes_Omega',sp.simplify(omega_y12-linear_omega)==0 and omega_y12!=0)
# Comparing naive frozen metric case (0) is wrong for h_yy(x)
gate('metric_sector_not_omittable',omega_y12.subs(dhdx,sp.Rational(1,10))==-sp.Rational(1,20))
# tetrad off-shell dimensions: at fixed g 4*4=16, 10 independent Lorentz constraints. 6 local boost+rotation
E0=sp.eye(4)
A4=sp.Matrix(4,4,lambda i,j:0)
for i in range(4):
    for j in range(i+1,4):
        xi=sp.symbols(f'x{i}{j}')
        A4[i,j]=xi
        A4[j,i]=-eta[i,i]*eta[j,j]*xi
# A4.T eta+eta A4=0
linearconstraint=sp.simplify(A4.T*eta+eta*A4)
gate('six_Lorentz_generators_preserve_metric_to_first_order',linearconstraint==sp.zeros(4))
gate('six_free_tetrad_Lorentz_parameters',len(A4.free_symbols)==6)
# raw full constrained algebra and Einstein metric evolution deliberately not solved
scientific_status={
 'CHH12_PROVENANCE_VERIFIED':True,
 'KOHLER_TIME_SPIN_PRIOR_ART_CONFIRMED':True,
 'SPATIAL_OPERATOR_DIFFERS_FROM_KOHLER_EQ17_OFF_SHELL':True,
 'GLOBAL_NOVELTY_PROVED':False,
 'ORTHONORMAL_TETRAD_CONSTRAINTS_EXACT':True,
 'AETHER_TRIAD_COUPLED_FIXED_METRIC_ACTION_DERIVED':True,
 'RESTRICTED_EOM_DERIVED_AND_TESTED':True,
 'RESTRICTED_COUPLED_NUMERICAL_ENERGY_CONSERVED':True,
 'FULL_COVARIANT_METRIC_EOM_EXPLICITLY_SOLVED':False,
 'FULL_DIRAC_CONSTRAINT_ALGEBRA_DERIVED':False,
 'FULL_COUPLED_GRAVITY_AETHER_SPIN_MODES_DERIVED':False,
 'STABLE_PHYSICAL_GRAVITY_MODE_PROVED':False,
 'INDEPENDENT_SI_PREDICTION':False,
 'EXPERIMENTAL_VALIDATION':False,
 'GVH_USED_IN_ALMA':False
}
gate('novelty_and_full_gravity_gates_remain_open',not scientific_status['GLOBAL_NOVELTY_PROVED'] and not scientific_status['FULL_COVARIANT_METRIC_EOM_EXPLICITLY_SOLVED'])
gate('metrics_not_misrepresented_as_predictions',not scientific_status['INDEPENDENT_SI_PREDICTION'] and not scientific_status['EXPERIMENTAL_VALIDATION'])

True

# Niveau 4 — Scientific verdict and next exact lock

**Result 1 (prior art):** temporal rotation dynamics of the triad is already in Kohler (2021), including couplings to other æther invariants and weak-field mode analysis.

**Result 2 (restricted operator comparison):** the spatial-gradient term in $S_{12}$ is genuinely different from Kohler's **particular** Eq. (17) under the same field identification; a periodic spatially twisting triad with constant $u,g$ distinguishes their off-shell actions. It is not thereby shown to be original to all physics, dynamically inequivalent after constraints, or to have detectable effects.

**Result 3 (coupled audit):** $u$ and triad jointly evolve in an exactly orthonormal, fixed-metric sector, with no externally prescribed shear. Energy and Euler–Lagrange tests pass. The gravitational stress tensor, metric EOM, full constraint Hamiltonian and all propagating mode norms remain open.

**Next decisive lock:** `GVH_CHH_14_Full_Metric_Spin_Principal_Symbol_and_Dirac_Constraints_GATE_FAST`: expand $g,u,e_A$ jointly about Minkowski with a consistent choice of tetrad parameterization; derive the complete quadratic action including Einstein-Hilbert, æther and spatial-spin terms; identify diffeomorphism constraints and physical propagating modes; compare with Kohler and existing triad/elastic EFTs. Do **not** treat this as an observational prediction until the constraints and matter coupling are addressed.

In [8]:
assert len(checks)==len(set(checks))
assert all(v['passed'] for v in checks.values())
result={
 'notebook':STEM,
 'date':'2026-11-18',
 'classification':'CANDIDATE_EXTENDED_ACTION_COUPLED_RESTRICTED_SECTOR_AND_KOHLER_EQ17_COMPARISON',
 'run_status':'PASS_LOCAL_TESTS; FULL_GRAVITY_GATE_OPEN',
 'check_count':len(checks), 'checks_pass':all(c['passed'] for c in checks.values()),
 'checks':checks,
 'scientific_status':scientific_status,
 'provenance':{'prior_executed_sha256':{k:v for k,v in metrics.items() if 'sha256' in k},'kohler_arxiv':'https://arxiv.org/abs/2109.03700','kohler_equation':'Eq. (17)','previous_model':'CHH12 extension, not original S_cand'},
 'assumptions':{'signature':'(-,+,+,+)','gravity':'fixed Minkowski metric, NOT Einstein equation solved','configuration':'boost beta(t,x) plus spatial rotation alpha(t,x)','orthonormality':'exact by construction','shear':'computed from u, not externally specified','parameter_values_prescribed_not_measured':{str(a):float(v) for a,v in par.items()},'matter_coupling':'not specified','units':'dimensionless c=1'},
 'metrics':{k:float(v) if isinstance(v,np.floating) else v for k,v in metrics.items()},
 'theory':{'reduced_Lagrangian':metrics['reduced_Lagrangian'],'angular_connection':'Omega_mu_12=-partial_mu alpha in restricted tetrad','directional_shear':'O_C=theta²*(2/3-(1/2)sin²(2alpha))','principal_symbol_reduced':'beta_speed²=(c1+c2+c3-zeta*f/2)/(c1+c4); spin_speed²=ks/kt','metric_coupling':'Omega_y12=-1/2 partial_x h_yy to first order'},
 'scientific_verdict':'COVARIANT_EXTENSION_POSTULATED; RESTRICTED_COUPLED_CONSTRAINT_AND_EOM_PASS; SPATIAL_OPERATOR_DIFFERS_FROM_KOHLER_2021_EQ17_ON_FIXED_FIELDS; FULL_PHYSICAL_INEQUIVALENCE_NOVELTY_STABILITY_AND_REAL_DATA_UNRESOLVED',
 'next_gate':'GVH_CHH_14_Full_Metric_Spin_Principal_Symbol_and_Dirac_Constraints_GATE_FAST'
}
result_path=BASE/(STEM+'_results.json')
result_path.write_text(json.dumps(result,indent=2,ensure_ascii=False)+'\n')
print('CHH13 RESULTS',json.dumps({'checks':len(checks),'passed':all(x['passed'] for x in checks.values()),'energy_drift':metrics['homogeneous_energy_max_relative_drift'],'delta_alpha':metrics['homogeneous_delta_alpha_zeta_vs_zero_at_t14'],'principal_speeds_squared':metrics['flat_restricted_principal_speeds_squared'],'full_gravity_gate':'OPEN'},indent=2))

CHH13 RESULTS {
  "checks": 65,
  "passed": true,
  "energy_drift": 1.979216790459759e-11,
  "delta_alpha": -0.01938753259709225,
  "principal_speeds_squared": {
    "beta": 0.6791666666666668,
    "alpha": 0.5
  },
  "full_gravity_gate": "OPEN"
}
